# E91 Entanglement-Based QKD

Alice and Bob share Bell pairs and measure in randomly chosen bases.
After public comparison, they extract a key and verify security
via CHSH-type correlations.

In [ ]:
import random
import qiskit as qk
import qiskit_aer as qka

NUM_PAIRS = 20

## Bell pair preparation and measurement

In [ ]:
def create_bell_pair():
    qc = qk.QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    return qc


def alice_measure(qc, basis):
    if basis == 1:
        qc.ry(3 * 3.14159 / 4, 0)
    elif basis == 2:
        qc.ry(3.14159 / 4, 0)
    qc.measure(0, 0)
    backend = qka.AerSimulator()
    compiled = qk.transpile(qc, backend)
    counts = backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0], 2)


def bob_measure(qc, basis):
    if basis == 1:
        qc.ry(3 * 3.14159 / 4, 1)
    elif basis == 2:
        qc.ry(3.14159 / 4, 1)
    qc.measure(1, 1)
    backend = qka.AerSimulator()
    compiled = qk.transpile(qc, backend)
    counts = backend.run(compiled, shots=1).result().get_counts()
    return int(list(counts.keys())[0], 2)

## Run E91 protocol

In [ ]:
alice_bases = [random.choice([0, 1, 2]) for _ in range(NUM_PAIRS)]
bob_bases = [random.choice([0, 1, 2]) for _ in range(NUM_PAIRS)]

alice_results, bob_results = [], []
for i in range(NUM_PAIRS):
    qc = create_bell_pair()
    alice_results.append(alice_measure(qc, alice_bases[i]))
    bob_results.append(bob_measure(qc, bob_bases[i]))

matching = [(i, a) for i, (ab, bb, a) in enumerate(zip(alice_bases, bob_bases, alice_results)) if ab == bb]
print(f"Matching bases: {len(matching)}/{NUM_PAIRS}")

key_a = [alice_results[i] for i, _ in matching]
key_b = [bob_results[i] for i, _ in matching]
print(f"Key A: {key_a}")
print(f"Key B: {key_b}")

check = random.sample(range(len(key_a)), min(4, len(key_a)))
errors = sum(1 for i in check if key_a[i] != key_b[i])
print(f"QBER: {errors}/{len(check)} = {errors / len(check):.2%}")